KPI-1 Encounter Mix by Encounter Class

In [0]:
%sql
with calendar as (
  select
    *
  from
    gold_medical_data.azure_blob_storage.dim_calendar
),
encounters as (
  select
    *
  from
    gold_medical_data.azure_blob_storage.dim_encounters
)
select
  year(e.encounter_start) as encounter_year,
  c.quater,
  e.encounter_class,
  count(*) as total_encounters,
  round(
    count(*) / (sum(count(*)) over (partition by c.quater)) * 100,
    2
  ) as total_encounters_percentage
from
  encounters e
    join calendar c
      on month(e.encounter_start) = c.month
group by
  encounter_year,
  c.quater,
  e.encounter_class
order by
  encounter_year,
  c.quater,
  e.encounter_class;

KPI-2 Encounters Over 24 Hours vs Under 24 Hours (%)

In [0]:
%sql
with encounter_durations AS (
  select
    date_format(encounter_start, 'yyyy-MM') as encounter_month,
    case
      when
        (unix_timestamp(encounter_stop) - unix_timestamp(encounter_start)) / 3600 > 24
      then
        'More than 24 hours'
      else '24 hours or less'
    end AS duration_category
  from
    gold_medical_data.azure_blob_storage.dim_encounters
  where
    encounter_start is not null
    and encounter_stop is not null
)
select
  encounter_month,
  duration_category,
  count(*) AS encounter_count,
  round(100.0 * encounter_count / sum(count(*)) over (partition by encounter_month), 2) as percentage
from
  encounter_durations
group by
  encounter_month,
  duration_category
order by
  encounter_month asc,
  duration_category;

KPI-3 Zero Payer Coverage Rate  

In [0]:
%sql
with fact_encounters as (
  select
    encounter_id,
    payer_coverage,
    case when payer_coverage > 0 then 1 else 0 end as has_payer_coverage
  from
    gold_medical_data.azure_blob_storage.fact_encounters
),
dim_encounters as (
  select
    encounter_id,
    date_format(encounter_stop, 'yyyy-MM') as encounter_month
  from
    gold_medical_data.azure_blob_storage.dim_encounters
)
select
  d.encounter_month,
  f.has_payer_coverage,
  count(*) as total_encounters,
  round(
    total_encounters / (sum(count(*)) over (partition by d.encounter_month)) * 100,
    2
  ) as total_encounters_percentage
from
  dim_encounters d
    join fact_encounters f
      on d.encounter_id = f.encounter_id
group by
  d.encounter_month,
  f.has_payer_coverage
order by
  d.encounter_month,
  f.has_payer_coverage
;

KPI-4 Top 10 Procedures by Highest Average Base Cost

In [0]:
%sql
select encounter_id,procedure_code,count(*) from gold_medical_data.azure_blob_storage.dim_procedures group by encounter_id,procedure_code order by encounter_id,procedure_code;
-- select * from gold_medical_data.azure_blob_storage.fact_procedures order by encounter_id,base_procedure_cost;